# GP-prior mock light curves with current PGMUVI parameter workflows

This tutorial draws reproducible one-dimensional mock light curves from three
explicit Gaussian-process priors:

- a quasi-periodic kernel;
- a Matérn stochastic/red-noise kernel; and
- a two-component spectral-mixture kernel.

The workflow uses the current model parameter schemas and
`apply_parameter_estimates`; it never writes GPyTorch `raw_*` parameters.


## Scope and boundaries

This is an advanced **prior-sampling** workflow. It does not condition on an
observed light curve, call `Lightcurve.fit`, run an optimizer, or produce a
posterior predictive distribution.

For deterministic analytic sinusoids, use
[`tutorial_synthetic.ipynb`](tutorial_synthetic.ipynb). For fitting and
consensus initialization, use the single-source fitting guide after generating
the mock data.


In [ ]:
import numpy as np
import torch
import gpytorch
import matplotlib.pyplot as plt

from pgmuvi.dtypes import DEFAULT_DTYPE
from pgmuvi.gps import MaternGPModel
from pgmuvi.gps import QuasiPeriodicGPModel
from pgmuvi.gps import SpectralMixtureGPModel
from pgmuvi.lightcurve import Lightcurve
from pgmuvi.parameter_estimates import ParameterEstimate
from pgmuvi.parameter_estimates import ParameterEstimateCollection
from pgmuvi.parameter_workflow import apply_parameter_estimates

SEED = 20260715
FIT_STARTED = False

torch.set_default_dtype(DEFAULT_DTYPE)
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"PGMUVI default dtype: {DEFAULT_DTYPE}")
print(f"sampling seed: {SEED}")
print(f"FIT_STARTED: {FIT_STARTED}")


## Apply user-specified physical values through the shared parameter layer

Each direct model exposes `parameter_schema()`. The helper below uses that
schema to create physical-space `ParameterEstimate` objects and then applies
them with `apply_parameter_estimates`.

This preserves the package's value transformations and shape checks. The
application report is retained as part of the mock-data provenance.


In [ ]:
def user_estimates(model, values):
    schema = model.parameter_schema()
    unknown = sorted(set(values) - set(schema.names()))
    if unknown:
        raise KeyError(f"Unknown parameter name(s): {unknown}")

    return ParameterEstimateCollection(
        [
            ParameterEstimate(
                spec=schema[name],
                value=value,
                value_source="tutorial_user_specified",
            )
            for name, value in values.items()
        ]
    )


def configure_model(model, values):
    report = apply_parameter_estimates(model, user_estimates(model, values))
    assert report is not None
    assert all(item["value"] for item in report.values())
    return report


def sample_latent_prior(model, x, seed):
    model.eval()
    torch.manual_seed(seed)
    with torch.no_grad():
        return model.forward(x).sample()


def observed_lightcurve(x, latent, noise_sigma, seed, name):
    torch.manual_seed(seed)
    yerr = torch.full_like(latent, noise_sigma)
    observed = latent + noise_sigma * torch.randn_like(latent)
    return Lightcurve(
        x,
        observed,
        yerr=yerr,
        max_samples=None,
        check_sampling=False,
        name=name,
    )


## Shared irregular observation times

All three examples use the same finite irregular time grid so the visual
differences are driven by the covariance kernels rather than by cadence.
The direct `ExactGP` constructors require placeholder target values. Those
placeholder values are not used as observations because the latent distribution
is requested directly from `model.forward(times)`.


In [ ]:
rng = np.random.default_rng(SEED)
time_values = np.sort(rng.uniform(0.0, 720.0, 72))
times = torch.as_tensor(time_values, dtype=DEFAULT_DTYPE)
placeholder = torch.sin(2.0 * torch.pi * times / 180.0)
NOISE_SIGMA = 0.12

print(f"rows: {times.numel()}")
print(f"time span: {float(times.max() - times.min()):.3f} days")


## 1. Quasi-periodic prior

The quasi-periodic model combines a periodic kernel with a long-term RBF decay.
The injected values are a 180-day period, a 650-day coherence timescale, and an
output variance of 1.4.


In [ ]:
qp_truth = {
    "period_days": 180.0,
    "coherence_days": 650.0,
    "output_variance": 1.4,
}
qp_likelihood = gpytorch.likelihoods.GaussianLikelihood()
qp_model = QuasiPeriodicGPModel(
    times,
    placeholder,
    qp_likelihood,
    period=qp_truth["period_days"],
)
qp_values = {
    "covar_module.outputscale": qp_truth["output_variance"],
    "covar_module.base_kernel.kernels.0.period_length": qp_truth[
        "period_days"
    ],
    "covar_module.base_kernel.kernels.1.lengthscale": qp_truth[
        "coherence_days"
    ],
}
qp_application_report = configure_model(qp_model, qp_values)
qp_latent = sample_latent_prior(qp_model, times, SEED + 101)
lc_qp = observed_lightcurve(
    times,
    qp_latent,
    NOISE_SIGMA,
    SEED + 201,
    "quasi_periodic_gp_prior_mock",
)

qp_period = float(
    qp_model.covar_module.base_kernel.kernels[0].period_length.squeeze()
)
print(f"quasi-periodic injected period: {qp_period:.3f} days")
print(f"quasi-periodic latent std: {float(qp_latent.std(unbiased=False)):.3f}")


## 2. Matérn stochastic prior

The Matérn model represents aperiodic stochastic/red-noise variability. Here
`nu=1.5`, the time-domain lengthscale is 85 days, and the output variance is
1.8. A Matérn realization should not be interpreted as having an injected
period.


In [ ]:
matern_truth = {
    "nu": 1.5,
    "lengthscale_days": 85.0,
    "output_variance": 1.8,
}
matern_likelihood = gpytorch.likelihoods.GaussianLikelihood()
matern_model = MaternGPModel(
    times,
    placeholder,
    matern_likelihood,
    nu=matern_truth["nu"],
    lengthscale=matern_truth["lengthscale_days"],
)
matern_values = {
    "covar_module.outputscale": matern_truth["output_variance"],
    "covar_module.base_kernel.lengthscale": matern_truth[
        "lengthscale_days"
    ],
}
matern_application_report = configure_model(matern_model, matern_values)
matern_latent = sample_latent_prior(matern_model, times, SEED + 102)
lc_matern = observed_lightcurve(
    times,
    matern_latent,
    NOISE_SIGMA,
    SEED + 202,
    "matern_gp_prior_mock",
)

matern_lengthscale = float(
    matern_model.covar_module.base_kernel.lengthscale.squeeze()
)
print(f"Matérn injected lengthscale: {matern_lengthscale:.3f} days")
print(f"Matérn latent std: {float(matern_latent.std(unbiased=False)):.3f}")


## 3. Spectral-mixture prior

The spectral-mixture parameters live in frequency space. The two injected
periods below are converted with `frequency = 1 / period` before constructing
the parameter estimates. `mixture_scales` are positive frequency widths and
`mixture_weights` are positive variance contributions.


In [ ]:
sm_periods = [180.0, 65.0]
sm_truth = {
    "periods_days": sm_periods,
    "frequencies_per_day": [1.0 / period for period in sm_periods],
    "frequency_widths_per_day": [7.0e-4, 2.8e-3],
    "variance_weights": [0.9, 0.35],
}
sm_likelihood = gpytorch.likelihoods.GaussianLikelihood()
sm_model = SpectralMixtureGPModel(
    times,
    placeholder,
    sm_likelihood,
    num_mixtures=2,
)
sm_values = {
    "covar_module.mixture_means": sm_truth["frequencies_per_day"],
    "covar_module.mixture_scales": sm_truth[
        "frequency_widths_per_day"
    ],
    "covar_module.mixture_weights": sm_truth["variance_weights"],
}
sm_application_report = configure_model(sm_model, sm_values)
sm_latent = sample_latent_prior(sm_model, times, SEED + 103)
lc_sm = observed_lightcurve(
    times,
    sm_latent,
    NOISE_SIGMA,
    SEED + 203,
    "spectral_mixture_gp_prior_mock",
)

sm_frequencies = (
    sm_model.covar_module.mixture_means.detach().reshape(-1).tolist()
)
print(f"spectral-mixture injected periods: {sm_periods}")
print(f"spectral-mixture frequencies: {sm_frequencies}")
print(f"spectral-mixture latent std: {float(sm_latent.std(unbiased=False)):.3f}")


## Compare the finite realizations

A kernel specifies a distribution over functions, not a single waveform.
Finite irregular samples can therefore look quite different from one seed to
another. The plotted points include independent observational noise; the thin
lines show the latent draws at the same timestamps.


In [ ]:
curves = [
    ("Quasi-periodic", lc_qp, qp_latent),
    ("Matérn", lc_matern, matern_latent),
    ("Spectral mixture", lc_sm, sm_latent),
]
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
for axis, (title, lightcurve, latent) in zip(axes, curves):
    axis.plot(times.numpy(), latent.numpy(), linewidth=1.2, label="latent prior")
    axis.errorbar(
        lightcurve.xdata.numpy(),
        lightcurve.ydata.numpy(),
        yerr=lightcurve.yerr.numpy(),
        fmt=".",
        alpha=0.65,
        label="noisy observations",
    )
    axis.set_title(title)
    axis.set_ylabel("flux")
    axis.legend()
axes[-1].set_xlabel("time [days]")
fig.tight_layout()
fig


## Preserve injected truth and parameter-application provenance

A recovery experiment needs more than the sampled flux. Record the physical
kernel values, the observation grid, both random seeds, the observational noise,
the dtype, and the parameter-application report.


In [ ]:
application_reports = {
    "quasi_periodic": qp_application_report,
    "matern": matern_application_report,
    "spectral_mixture": sm_application_report,
}
injected_truth = {
    "quasi_periodic": qp_truth,
    "matern": matern_truth,
    "spectral_mixture": sm_truth,
}
all_values_applied = all(
    item["value"]
    for report in application_reports.values()
    for item in report.values()
)

print(f"all requested physical values applied: {all_values_applied}")
print(f"stored truth families: {sorted(injected_truth)}")
print(f"default transform on mock Lightcurves: {type(lc_qp.xtransform).__name__}")


## Multiple realizations are more informative than one favorable draw

The next cell draws three independent Matérn realizations from the same prior.
Use ensembles like this when testing recovery rates, false positives, or the
effect of cadence and baseline.


In [ ]:
matern_model.eval()
torch.manual_seed(SEED + 500)
with torch.no_grad():
    matern_ensemble = matern_model.forward(times).sample(torch.Size([3]))

ensemble_shape = tuple(matern_ensemble.shape)
print(f"Matérn ensemble shape: {ensemble_shape}")


## Posterior predictive samples are a different workflow

Posterior predictive samples require a successfully fitted model and its
likelihood. They are conditioned on observations and should not be described as
GP-prior mocks. This notebook deliberately stops before fitting and does not use
private evaluation helpers.

To fit one of the generated `Lightcurve` objects, continue with the maintained
single-source and consensus-fitting documentation. To interpret the fitted
period, PSD peaks, ARD diagnostics, or failures, use the result-interpretation
guide.


## Reproducibility checklist

Record all of the following for a scientific mock-data experiment:

1. model/kernel family and all physical hyperparameters;
2. observation times and time units;
3. Torch and NumPy seeds;
4. observational-noise distribution and scale;
5. package dtype and revision;
6. latent realization when exact comparisons are required; and
7. the complete fit configuration used later for recovery.

The corresponding command-line example is `examples/gp_prior_sampling.py`.


## Summary

- `pgmuvi.synthetic` provides deterministic analytic signals.
- This notebook provides random latent GP-prior realizations.
- `apply_parameter_estimates` applies physical values without direct raw-parameter writes.
- Measurement noise is added explicitly before constructing `Lightcurve` objects.
- No fit, optimizer, posterior, or automatic model selection is performed.
